# Notebook 02: Exploratory analysis & correlation

## How this fits with Notebook 01

Notebook 01 produced **`country_year_modeling_panel_cleaned.csv`**: imputed predictors and `log_mmr`. This notebook **explores that analysis-ready file**—summaries, correlations, and a few key plots—so you can describe patterns in the data before fitting models (Notebook 03).

**Do not skip Notebook 01** unless you already have an equivalent cleaned file at the path below.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/processed/country_year_modeling_panel_cleaned.csv"
df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [
    "gdp_pc", "health_exp_gdp", "fertility", "skilled_births",
    "pm25", "heat_index35_days", "cooling_degree_days",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

print("Shape:", df.shape)
df.head()


## Descriptive statistics

One table for **all numeric columns** (quick global summary). For write-ups, you can highlight the modeling block: features + `mmr` + `log_mmr`.


In [ ]:
from IPython.display import display

# All numeric columns
display(df.select_dtypes(include=[np.number]).describe())

# Modeling-focused subset (same features as later notebooks + outcomes)
modeling_numeric = [c for c in FEATURE_COLS + ["mmr", "log_mmr"] if c in df.columns]
display(df[modeling_numeric].describe())


## Correlation matrix (Pearson)

Shows **linear** association among predictors and raw `mmr`. After Notebook 01, modeling features should have **no missing values**; this uses those columns directly (no extra median fill for the main heatmap). If any NaN remains, the next cell will raise a clear error.


In [ ]:
cols = FEATURE_COLS + ["mmr"]
assert df[cols].isna().sum().sum() == 0, (
    "Unexpected missing values in correlation columns — re-run Notebook 01 or check FEATURE_COLS."
)

corr = df[cols].corr(method="pearson")

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, center=0)
plt.title("Pearson correlation: modeling features and MMR")
plt.tight_layout()
plt.show()


## Bivariate plots (raw MMR)

These complement the correlation matrix: **fertility** and **GDP per capita** vs **MMR** (actual deaths per 100k scale). Color encodes GDP on the fertility plot to show a third dimension at a glance.


In [ ]:
plt.figure(figsize=(7, 5))
sc = plt.scatter(df["fertility"], df["mmr"], c=df["gdp_pc"], cmap="viridis", alpha=0.55)
plt.colorbar(sc, label="GDP per capita")
plt.xlabel("Fertility rate (births per woman)")
plt.ylabel("Maternal mortality ratio (MMR)")
plt.title("Fertility vs. MMR (color = GDP per capita)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(df["gdp_pc"], df["mmr"], alpha=0.45)
plt.xlabel("GDP per capita")
plt.ylabel("Maternal mortality ratio (MMR)")
plt.title("GDP per capita vs. MMR")
plt.tight_layout()
plt.show()
